# Module 11: Fine-tuning language models responsibly

Perform real parameter-efficient gradient updates on annotated messages, preserve the base model, and audit held-out loss and replay.

Use the complete source folder and the documented isolated CPU framework environment. Run in order, retain named text/tables, save, restart, and run all. First source/model downloads need internet; valid caches support replay. No earlier notebook state or paid account is needed.

The public script calls the same implementation. Small-model mechanics and valid software execution do not establish task quality or accessible notebook-interface operation.


In [ ]:
from pathlib import Path
import sys
import json
import tempfile
import platform
import importlib.metadata
import numpy as np
import pandas as pd

# Find shared code from the complete checkout/archive, not copied notebook logic.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "ml_ai_it"]
ROOT = next((p for p in candidates if (p / "src/practical_ml_ai").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete ml_ai_it source folder.")
sys.path.insert(0, str(ROOT / "src"))
from practical_ml_ai.course_runner import run_module

# A new directory allows clean replay without silently replacing frozen evidence.
(ROOT / "runs").mkdir(exist_ok=True)
parent = Path(tempfile.mkdtemp(prefix="module11-notebook-", dir=ROOT / "runs"))
run_dir = parent / "study"
print("Evidence will be saved to:", run_dir)


## Inspect the real source and boundaries

These are actual human-labeled SMS records. Exact normalized texts do not cross splits; reports use hashes rather than raw messages.

The default uses eight actual gradient steps, alternating real ham/spam training records. Validation and final subsets are disjoint text IDs. Only adapter parameters are optimized; base weights stay frozen. Base/adapted final completion loss comparisons are predefined. Numeric adapter arrays and configuration are saved and reloaded, with a replay check. No synthetic labels/messages are generated for training.

Before running, identify the observation unit, input information, and fitting/selection boundary. No generated dataset or fabricated labels substitute for the external source.


In [ ]:
from practical_ml_ai.continuation_data import sms_splits
split, source_manifest = sms_splits()
for name, part in split.items():
    # Display IDs/counts rather than raw messages that may contain personal data.
    print(name, "rows:", len(part), "positive labels:", int(part.label.sum()))
print("First training text IDs:", split["train"].text_id.head(3).tolist())
print("Sampling rule:", source_manifest["sampling"])


## Predict and inspect the mechanism

A frozen base still performs the forward computation. Gradients flow to the low-rank matrices. Starting one matrix at zero preserves the original output initially; an optimizer step can then change the adapter. A nonzero update proves mechanics, not improved language quality.

LoRA adds a low-rank update to a frozen linear projection. Here a rank-2 pair targets one query projection; this is a deliberately small adaptation, not full-model retraining. Masking prompt labels with -100 prevents the objective from training on every instruction token.


In [ ]:
from practical_ml_ai.language import local_model, packed_example
from practical_ml_ai.continuation_data import balanced_real_subset
model, tokenizer, revision = local_model()
real_training_row = balanced_real_subset(split["train"], 24).iloc[0]
input_tokens, loss_labels = packed_example(tokenizer, real_training_row)
print("Actual training text ID:", real_training_row.text_id)
print("Token shape:", tuple(input_tokens.shape))
print("Masked prompt positions:", int((loss_labels == -100).sum()))
print("Supervised completion positions:", int((loss_labels != -100).sum()))
# Do not train on validation/final labels or accidentally unmask the whole prompt.


## Run the shared workflow

The script and this cell call one implementation. Training/validation choices and default compute budgets are fixed before the final report. A source hash mismatch or completed-directory error should be investigated, not bypassed.

Eight actual adapter-gradient steps are a mechanics check; the code does not claim that token loss establishes safe behavior.


In [ ]:
# Actual execution, not a mocked result. Change optional budgets only as a declared development experiment.
result = run_module(11, run_dir, steps=8)
print((run_dir / "summary.txt").read_text(encoding="utf-8"))


## Inspect the exact tables

The core checks actual gradients and numeric replay; it does not replace training with a mocked response. Tiny balanced samples and a short step budget cannot validate generalization. An unchanged/worse final loss is a result to report. Repeatedly changing steps after viewing final loss would invalidate its holdout role.

The first rows below are a preview. Open the full CSV/HTML report to inspect all values. Tables preserve the evidence without requiring a sight-dependent chart or screenshot.


In [ ]:
table = pd.read_csv(run_dir / "training_history.csv")
print("Evidence table: training_history.csv")
print(table.head(15).to_string(index=False))
print("Complete report:", run_dir / "index.html")


## Verify one interpretation

Use the actual output to check a denominator, fitting boundary, update, or budget. A passing software check is separate from whether the method is useful.


In [ ]:
adapter_state = np.load(run_dir / "adapter.npz", allow_pickle=False)
print("Adapter matrix shapes:", adapter_state["a"].shape, adapter_state["b"].shape)
print("Nonzero update norm:", float(np.linalg.norm(adapter_state["b"])))
print("Base/adapted final completion loss:",
      result["base_test_completion_loss"], result["adapted_test_completion_loss"])
print("Saved-adapter replay:", result["adapter_replay_verified"])


## Record versions and save

Retain the source/protocol manifest, output directory, commands, versions, and your own interpretation. Complete the lesson's practice and reflection; optional depth is skippable. Restart-and-run-all makes hidden state visible, but repeated public final data are not fresh independent evaluation.


In [ ]:
versions = {"python": platform.python_version()}
for package in ["numpy","pandas","torch","transformers"]:
    versions[package] = importlib.metadata.version(package)
(run_dir / "environment.json").write_text(json.dumps(versions, indent=2) + "\n", encoding="utf-8")
print("Reproduction versions:", versions)
print("Your saved evidence:", run_dir)


## Independent practice

Identify trainable parameters and masked token positions. Verify a nonzero adapter update, exact source/revision, disjoint IDs, held-out losses, and saved-adapter replay. Explain why a lower token loss does not prove safe, factual, or accurate task output.

Optional: Use a predeclared validation plan for another rank, step budget, or target projection. More complete supervised training and a credible prospective evaluation require additional real data, compute, and review; they are optional depth.

Explain one limitation and a next action. If using GenAI, verify its suggestions against actual code/tables and follow local disclosure rules. Instructors can adapt the linked practice criteria while preserving learning goals and accessibility.
